# Does wage growth predict US inflation?

**Questions**
1. Does wage growth help predict core PCE inflation once energy prices, labor-market slack and inflation expectations are accounted for?
2. Do models that use wage growth forecast core PCE inflation better *out of sample* than simple benchmarks?

**Approach.** Lead/lag (Granger) tests, a Phillips-curve regression with HAC standard errors, and an expanding-window backtest with an explicit embargo so no model sees outcomes that were not yet known. Everything here is predictive; none of it identifies causal effects.

**Data.** Monthly FRED series (see `src/fetch.py`). Requires a free API key in the `FRED_API_KEY` environment variable.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd

from src import models
from src.analysis import derive
from src.fetch import build_dataset

warnings.filterwarnings("ignore", category=UserWarning)
pd.options.display.float_format = "{:.3f}".format

HORIZON = 6   # forecast core PCE inflation over the next 6 months
raw = build_dataset()
df = derive(raw)
print(df.index.min().date(), "to", df.index.max().date(), f"({len(df)} months)")
df[["core_pce", "wages", "energy_cpi", "unrate", "infl_exp"]].tail()

## 1. Data checks

Missing months matter: the 2025 federal shutdown left October 2025 out of the CPI and household-survey series. Core PCE, the target, was not affected.

In [ ]:
cols = ["core_pce", "wages", "energy_cpi", "unrate", "infl_exp"]
missing = df[cols].isna()
print("Months with a missing model input:")
print(missing[missing.any(axis=1)].sum(axis=0)[lambda s: s > 0])

frame = models.build_features(df, horizon=HORIZON)
print("\nUsable rows after building features:", len(frame.dropna(subset=models.FULL_FEATURES + ["target"])))
models.adf_table(frame, ["infl_1m", "wage_1m"])

## 2. Do wages lead inflation? (Granger tests)

Low p-values mean the first series helps predict the second beyond its own past. The reverse direction tests the "wage-price spiral" feedback. Series that fail a stationarity test are first-differenced first.

In [ ]:
models.granger_table(frame, maxlag=6)

## 3. Phillips-curve regression

Future core inflation regressed on current inflation, energy prices, unemployment, expectations and wage growth. Forecast windows overlap, so residuals are autocorrelated and the standard errors are HAC (Newey-West).

In [ ]:
table, summary = models.phillips_regression(frame, horizon=HORIZON)
print(summary)
table

In [ ]:
# Is the role of wage growth stable before and after 2021?
models.regime_comparison(frame, horizon=HORIZON)

## 4. Out-of-sample backtest

Models are refit every month on rows whose outcomes were already observed at that time (`train_mask`). Benchmarks: trailing 12-month inflation (naive), an autoregressive ridge model, ridge with energy/slack/expectations, ridge with wage growth added, and gradient boosting. Accuracy is compared with RMSE, MAE and a Diebold-Mariano test against the naive benchmark.

In [ ]:
preds = models.walk_forward(frame, horizon=HORIZON)
scores = models.score_periods(preds, HORIZON)
for label, tbl in scores.items():
    print(label)
    display(tbl)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(preds.index, preds["actual"], color="black", lw=2.5, label="Actual")
for col in preds.columns.drop("actual"):
    ax.plot(preds.index, preds[col], lw=1.2, label=col)
ax.set_title(f"Walk-forward forecasts: core PCE inflation over the next {HORIZON} months (annualized %)")
ax.set_ylabel("%"); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Conclusions

_Write 3 to 4 sentences after running the notebook, based only on the tables and chart above: whether wage growth adds predictive information, whether the models beat the naive benchmark (and whether the Diebold-Mariano test says the difference is real), and one caveat about sample size or the 2021-22 inflation surge._

## Limitations

- About 15 years of monthly data and overlapping forecast windows mean few independent observations.
- The 2020-22 period is unusual and dominates error metrics, so results are also shown from 2023 onward.
- Granger tests and regressions show predictive relationships, not causal effects.
- Average hourly earnings are affected by composition changes (for example in spring 2020).
- October 2025 is missing from the CPI and household-survey series because of the federal shutdown.